# 07 — Évaluation Out-of-Domain : LIAR → WELFake (Classification Binaire)

## Contexte

Le notebook **06** a montré que la classification binaire (fake / real) est plus performante et plus cohérente que la classification 3 classes sur LIAR. La classe "nuanced" introduit une frontière floue et pénalise le F1 macro.

Ce notebook évalue la **capacité de généralisation** des modèles binaires entraînés sur LIAR vers le dataset WELFake — un dataset externe de référence.

## Pourquoi le binaire est indispensable pour l'évaluation OOD

WELFake est un dataset **nativement binaire** (fake / real, pas de classe nuanced). Évaluer des modèles 3 classes sur WELFake serait méthodologiquement incorrect : la classe "nuanced" aurait un support de 0, son F1 serait mécaniquement nul, et le F1 macro serait artificiellement tiré vers le bas — sans que ça reflète les vraies performances sur fake et real.

## Dataset externe : WELFake

| | LIAR (entraînement) | WELFake (évaluation) |
|---|---|---|
| Source | PolitiFact | Kaggle + McIntire + Reuters + BuzzFeed |
| Type de texte | Déclaration (~18 mots) | Article complet (~540 mots) |
| Labels | **Binaire** (fake=0, real=1) | **Binaire** (fake=0, real=1) |
| Taille | 6 498 (train, après filtre nuanced) | 72 134 |
| Métadonnées speaker | **Oui** | **Non** |

## Hypothèses

1. **Les modèles avec métadonnées vont chuter davantage** — credibility_score et lie_rate sont à 0 pour tous les articles WELFake, ce qui biaise les prédictions vers "real".
2. **RF texte seul devrait mieux généraliser** — il ne dépend pas des métadonnées speaker absentes de WELFake.
3. **La chute sera moindre qu'avec les modèles 3 classes** — l'espace de labels est désormais aligné entre les deux datasets.

## 0. Imports

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from preprocessing import load_liar, preprocess_liar
from features import TfidfFeatures, CombinedFeatures, META_COLS
from train import train_logistic_regression, train_random_forest
from welfake_loader import load_welfake, preprocess_welfake, add_dummy_metadata

print("Imports OK.")

Imports OK.


## 1. Préparation des données LIAR — version binaire

On filtre les exemples "nuanced" et on remémorise : **fake = 0 / real = 1**

In [2]:
train_raw, valid_raw, test_raw = load_liar('data/raw')
train_full = preprocess_liar(train_raw)
test_full  = preprocess_liar(test_raw)

def make_binary(df):
    """Supprime les nuanced, remémorise fake=0 / real=1."""
    df_bin = df[df['label_encoded'] != 1].copy()
    df_bin['label_binary'] = df_bin['label_encoded'].map({0: 0, 2: 1})
    return df_bin

train = make_binary(train_full)
test  = make_binary(test_full)

y_train = train['label_binary'].values
y_test  = test['label_binary'].values

print(f"Train : {len(train)} exemples  |  Test : {len(test)} exemples")
print(f"Distribution train — fake: {(y_train==0).sum()}  real: {(y_train==1).sum()}")

LIAR chargé — train: 10240, valid: 1284, test: 1267
Train : 6472 exemples  |  Test : 790 exemples
Distribution train — fake: 2834  real: 3638


## 2. Extraction des features

In [3]:
# TF-IDF + métadonnées (pipeline principal)
tfidf    = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf.fit_transform(train['statement_clean'])
combined = CombinedFeatures(text_features=tfidf, meta_cols=META_COLS)
X_train  = combined.fit_transform(train)
X_test   = combined.transform(test)

# TF-IDF seul (pour mesurer l'impact des métadonnées en OOD)
X_train_text = tfidf.transform(train['statement_clean'])
X_test_text  = tfidf.transform(test['statement_clean'])

print(f"X_train (avec meta) : {X_train.shape}")
print(f"X_train (texte seul): {X_train_text.shape}")

TF-IDF fit — vocabulaire : 5912 features, 6472 documents
X_train (avec meta) : (6472, 5916)
X_train (texte seul): (6472, 5912)


## 3. Entraînement des modèles binaires sur LIAR

In [4]:
def train_xgboost_binary(X_train, y_train):
    """XGBoost avec objectif binaire."""
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    sample_weight = np.array([weights[y] for y in y_train])
    X_dense = X_train.toarray() if hasattr(X_train, "toarray") else X_train
    model = XGBClassifier(
        objective="binary:logistic", n_estimators=300, max_depth=6,
        learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", random_state=42, n_jobs=-1,
        eval_metric="logloss", verbosity=0,
    )
    model.fit(X_dense, y_train, sample_weight=sample_weight)
    return model

print("Entraînement des modèles avec métadonnées...")
lr      = train_logistic_regression(X_train, y_train)
rf      = train_random_forest(X_train, y_train, n_estimators=300)
xgb     = train_xgboost_binary(X_train, y_train)

print("\nEntraînement RF texte seul (référence sans métadonnées)...")
rf_text = train_random_forest(X_train_text, y_train, n_estimators=300)

print("\nTous les modèles entraînés.")

Entraînement des modèles avec métadonnées...
Entraînement Logistic Regression...
  LR — terminé
Entraînement Random Forest...
  RF — terminé

Entraînement RF texte seul (référence sans métadonnées)...
Entraînement Random Forest...
  RF — terminé

Tous les modèles entraînés.


## 4. Scores in-domain (référence LIAR binaire)

In [5]:
def eval_binary(y_true, y_pred, model_name):
    """Évaluation binaire (fake=0, real=1)."""
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    f1c = f1_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}  |  F1 macro : {f1:.4f}   ← métrique principale")
    print(f"\n  F1 par classe : fake={f1c[0]:.4f}  real={f1c[1]:.4f}")
    print(classification_report(y_true, y_pred, target_names=['fake','real'],
                                labels=[0,1], zero_division=0))
    return {
        'model': model_name, 'accuracy': round(acc,4), 'f1_macro': round(f1,4),
        'f1_fake': round(f1c[0],4), 'f1_real': round(f1c[1],4),
    }

X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else X_test

results_indomain = []
results_indomain.append(eval_binary(y_test, lr.predict(X_test),             'LR'))
results_indomain.append(eval_binary(y_test, rf.predict(X_test),             'RF'))
results_indomain.append(eval_binary(y_test, xgb.predict(X_test_dense),      'XGBoost'))
res_rf_text_in = eval_binary(y_test, rf_text.predict(X_test_text),          'RF texte seul')


  LR
  Accuracy : 0.7899  |  F1 macro : 0.7867   ← métrique principale

  F1 par classe : fake=0.7608  real=0.8126
              precision    recall  f1-score   support

        fake       0.75      0.77      0.76       341
        real       0.82      0.80      0.81       449

    accuracy                           0.79       790
   macro avg       0.79      0.79      0.79       790
weighted avg       0.79      0.79      0.79       790


  RF
  Accuracy : 0.7873  |  F1 macro : 0.7837   ← métrique principale

  F1 par classe : fake=0.7558  real=0.8117
              precision    recall  f1-score   support

        fake       0.75      0.76      0.76       341
        real       0.82      0.81      0.81       449

    accuracy                           0.79       790
   macro avg       0.78      0.78      0.78       790
weighted avg       0.79      0.79      0.79       790


  XGBoost
  Accuracy : 0.7747  |  F1 macro : 0.7723   ← métrique principale

  F1 par classe : fake=0.7493  real=

## 5. Chargement et preprocessing WELFake

In [6]:
welfake = load_welfake(
    path='data/external/WELFake_Dataset.csv',
    sample_n=5000,
    random_state=42,
)
welfake = preprocess_welfake(welfake, max_chars=500)
welfake = add_dummy_metadata(welfake)

# Labels WELFake alignés avec le binaire LIAR : fake=0, real=1
# (welfake_loader retourne label_encoded={0,2} → on remémorise 2→1)
y_welfake = welfake['label_encoded'].map({0: 0, 2: 1}).values

print(f"WELFake : {len(welfake)} articles")
print(f"Classes : {np.unique(y_welfake)}  (0=fake, 1=real)")
print(f"Distribution : fake={( y_welfake==0).sum()}  real={(y_welfake==1).sum()}")

WELFake chargé — 10000 articles
Distribution des labels :
  fake       :   5000  (50.0%)
  real       :   5000  (50.0%)
  14 textes vides après nettoyage
Preprocessing terminé — longueur moy. après nettoyage : 51 mots
WELFake : 10000 articles
Classes : [0 1]  (0=fake, 1=real)
Distribution : fake=5000  real=5000


## 6. Construction des features WELFake

On applique le TF-IDF **appris sur LIAR** (`.transform()` uniquement — pas de refit).  
C'est le cœur du test de généralisation.

In [7]:
X_welfake      = combined.transform(welfake)
X_welfake_text = tfidf.transform(welfake['statement_clean'])
X_welfake_dense = X_welfake.toarray() if hasattr(X_welfake, 'toarray') else X_welfake

# Couverture du vocabulaire LIAR sur WELFake
nonzero = (X_welfake_text.sum(axis=1) > 0).sum()
density_liar    = X_test_text.nnz / (X_test_text.shape[0] * X_test_text.shape[1])
density_welfake = X_welfake_text.nnz / (X_welfake_text.shape[0] * X_welfake_text.shape[1])

print(f"Couverture vocab LIAR sur WELFake : {nonzero}/{len(welfake)} ({nonzero/len(welfake)*100:.1f}%)")
print(f"Densité TF-IDF — LIAR: {density_liar*100:.4f}%  |  WELFake: {density_welfake*100:.4f}%")
print()
print("Note : la densité plus haute sur WELFake reflète la longueur des articles (540 mots vs 18).")
print("Le vocabulaire est connu mais la distribution TF-IDF est très différente.")

Couverture vocab LIAR sur WELFake : 9970/10000 (99.7%)
Densité TF-IDF — LIAR: 0.1881%  |  WELFake: 0.5158%

Note : la densité plus haute sur WELFake reflète la longueur des articles (540 mots vs 18).
Le vocabulaire est connu mais la distribution TF-IDF est très différente.


## 7. Évaluation out-of-domain sur WELFake

In [8]:
results_outdomain = []
results_outdomain.append(eval_binary(y_welfake, lr.predict(X_welfake),          'LR'))
results_outdomain.append(eval_binary(y_welfake, rf.predict(X_welfake),          'RF'))
results_outdomain.append(eval_binary(y_welfake, xgb.predict(X_welfake_dense),   'XGBoost'))
res_rf_text_out = eval_binary(y_welfake, rf_text.predict(X_welfake_text),       'RF texte seul')


  LR
  Accuracy : 0.5000  |  F1 macro : 0.3333   ← métrique principale

  F1 par classe : fake=0.0000  real=0.6667
              precision    recall  f1-score   support

        fake       0.00      0.00      0.00      5000
        real       0.50      1.00      0.67      5000

    accuracy                           0.50     10000
   macro avg       0.25      0.50      0.33     10000
weighted avg       0.25      0.50      0.33     10000


  RF
  Accuracy : 0.5000  |  F1 macro : 0.3333   ← métrique principale

  F1 par classe : fake=0.0000  real=0.6667
              precision    recall  f1-score   support

        fake       0.00      0.00      0.00      5000
        real       0.50      1.00      0.67      5000

    accuracy                           0.50     10000
   macro avg       0.25      0.50      0.33     10000
weighted avg       0.25      0.50      0.33     10000


  XGBoost
  Accuracy : 0.5004  |  F1 macro : 0.3344   ← métrique principale

  F1 par classe : fake=0.0020  real=

## 8. Analyse de la chute de performance

In [9]:
df_in  = pd.DataFrame(results_indomain)
df_out = pd.DataFrame(results_outdomain)

MODELS = ['LR', 'RF', 'XGBoost']

print(f"{'Modèle':<16} | {'In-domain':>10} | {'Out-domain':>10} | {'Chute':>8}")
print('-' * 55)
for m in MODELS:
    f1_in  = df_in[df_in['model']==m]['f1_macro'].values[0]
    f1_out = df_out[df_out['model']==m]['f1_macro'].values[0]
    print(f"  {m:<14} | {f1_in:>10.4f} | {f1_out:>10.4f} | {(f1_out-f1_in)*100:>+7.2f} pp")

print()
print("Impact des métadonnées sur la généralisation (RF) :")
rf_in  = df_in[df_in['model']=='RF']['f1_macro'].values[0]
rf_out = df_out[df_out['model']=='RF']['f1_macro'].values[0]
print(f"  RF avec méta  — in:{rf_in:.4f}  out:{rf_out:.4f}  chute:{(rf_out-rf_in)*100:+.2f} pp")
print(f"  RF texte seul — in:{res_rf_text_in['f1_macro']:.4f}  out:{res_rf_text_out['f1_macro']:.4f}  chute:{(res_rf_text_out['f1_macro']-res_rf_text_in['f1_macro'])*100:+.2f} pp")

Modèle           |  In-domain | Out-domain |    Chute
-------------------------------------------------------
  LR             |     0.7867 |     0.3333 |  -45.34 pp
  RF             |     0.7837 |     0.3333 |  -45.04 pp
  XGBoost        |     0.7723 |     0.3344 |  -43.79 pp

Impact des métadonnées sur la généralisation (RF) :
  RF avec méta  — in:0.7837  out:0.3333  chute:-45.04 pp
  RF texte seul — in:0.6151  out:0.4644  chute:-15.07 pp


## 9. Visualisations interactives (Plotly)

In [10]:
# --- Graphique 1 : F1 macro In vs Out par modèle ---
fig1 = go.Figure()
fig1.add_trace(go.Bar(
    name="In-domain (LIAR)", x=MODELS,
    y=[df_in[df_in['model']==m]['f1_macro'].values[0] for m in MODELS],
    marker_color="#378ADD", opacity=0.85,
    text=[f"{df_in[df_in['model']==m]['f1_macro'].values[0]:.3f}" for m in MODELS],
    textposition="outside",
))
fig1.add_trace(go.Bar(
    name="Out-of-domain (WELFake)", x=MODELS,
    y=[df_out[df_out['model']==m]['f1_macro'].values[0] for m in MODELS],
    marker_color="#D85A30", opacity=0.85,
    text=[f"{df_out[df_out['model']==m]['f1_macro'].values[0]:.3f}" for m in MODELS],
    textposition="outside",
))
fig1.update_layout(
    title=dict(text="F1 macro — In-domain (LIAR) vs Out-of-domain (WELFake)", font=dict(size=16)),
    barmode="group",
    yaxis=dict(title="F1 macro", range=[0, 1.0], tickformat=".2f"),
    xaxis_title="Modèle",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white", height=460,
)
fig1.show()

In [11]:
# --- Graphique 2 : Chute en points de pourcentage ---
all_models_with_rf_text = MODELS + ['RF texte seul']
f1_in_all  = [df_in[df_in['model']==m]['f1_macro'].values[0] for m in MODELS] + [res_rf_text_in['f1_macro']]
f1_out_all = [df_out[df_out['model']==m]['f1_macro'].values[0] for m in MODELS] + [res_rf_text_out['f1_macro']]
drops = [(m, (o-i)*100) for m, i, o in zip(all_models_with_rf_text, f1_in_all, f1_out_all)]
drops_sorted = sorted(drops, key=lambda x: x[1], reverse=True)

fig2 = go.Figure(go.Bar(
    x=[d[1] for d in drops_sorted],
    y=[d[0] for d in drops_sorted],
    orientation="h",
    marker_color=["#27ae60" if d[1] >= 0 else "#c0392b" for d in drops_sorted],
    opacity=0.85,
    text=[f"{d[1]:+.1f} pp" for d in drops_sorted],
    textposition="outside",
))
fig2.add_vline(x=0, line_color="black", line_width=1.5)
fig2.update_layout(
    title=dict(text="Chute de performance In-domain → Out-of-domain (F1 macro)", font=dict(size=16)),
    xaxis=dict(title="Chute (points de pourcentage)", tickformat="+.0f", range=[-55, 5]),
    yaxis_title="Modèle",
    template="plotly_white", height=380, margin=dict(l=130),
)
fig2.show()

In [12]:
# --- Graphique 3 : F1 fake et F1 real — In vs Out ---
fig3 = make_subplots(rows=1, cols=2,
    subplot_titles=["F1 — classe fake", "F1 — classe real"],
    shared_yaxes=True)

for col_idx, metric in enumerate(['f1_fake', 'f1_real'], start=1):
    vals_in  = [df_in[df_in['model']==m][metric].values[0] for m in MODELS]
    vals_out = [df_out[df_out['model']==m][metric].values[0] for m in MODELS]
    fig3.add_trace(go.Bar(
        name="In-domain" if col_idx==1 else None,
        showlegend=(col_idx==1), legendgroup="in",
        x=MODELS, y=vals_in, marker_color="#378ADD", opacity=0.85,
        text=[f"{v:.3f}" for v in vals_in], textposition="outside",
    ), row=1, col=col_idx)
    fig3.add_trace(go.Bar(
        name="Out-of-domain" if col_idx==1 else None,
        showlegend=(col_idx==1), legendgroup="out",
        x=MODELS, y=vals_out, marker_color="#D85A30", opacity=0.85,
        text=[f"{v:.3f}" for v in vals_out], textposition="outside",
    ), row=1, col=col_idx)

fig3.update_layout(
    title=dict(text="F1 par classe (fake / real) — In-domain vs Out-of-domain", font=dict(size=16)),
    barmode="group", yaxis=dict(title="F1", range=[0, 1.0], tickformat=".2f"),
    template="plotly_white", height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig3.show()

In [13]:
# --- Graphique 4 : Heatmap toutes métriques × modèles × domaines ---
ALL_METRICS   = ["accuracy", "f1_macro", "f1_fake", "f1_real"]
METRIC_LABELS = ["Accuracy", "F1 macro", "F1 fake", "F1 real"]
ALL_MODELS_EXTENDED = MODELS + ['RF texte seul']

rows_in  = {r['model']: r for r in results_indomain}
rows_in['RF texte seul'] = res_rf_text_in
rows_out = {r['model']: r for r in results_outdomain}
rows_out['RF texte seul'] = res_rf_text_out

row_labels, z_matrix = [], []
for m in ALL_MODELS_EXTENDED:
    for (row_dict, dom) in [(rows_in[m], "in-domain"), (rows_out[m], "out-of-domain")]:
        row_labels.append(f"{m} ({dom})")
        z_matrix.append([round(row_dict[k], 3) for k in ALL_METRICS])

fig4 = go.Figure(data=go.Heatmap(
    z=z_matrix, x=METRIC_LABELS, y=row_labels,
    colorscale="RdYlGn", zmin=0.0, zmax=0.85,
    text=[[f"{v:.3f}" for v in row] for row in z_matrix],
    texttemplate="%{text}", textfont=dict(size=11),
    showscale=True, colorbar=dict(title="Score"),
))
fig4.update_layout(
    title=dict(text="Heatmap — Toutes métriques × Modèles × Domaines", font=dict(size=16)),
    xaxis_title="Métrique",
    yaxis=dict(autorange="reversed", title="Modèle × Domaine"),
    template="plotly_white", height=560, margin=dict(l=220),
)
fig4.show()

## 10. Sauvegarde

In [14]:
rows = []
for r_in, r_out in zip(results_indomain, results_outdomain):
    m = r_in['model']
    rows.append(r_in  | {'domain': 'in-domain (LIAR)',       'mode': 'binaire'})
    rows.append(r_out | {'domain': 'out-of-domain (WELFake)', 'mode': 'binaire'})
rows.append(res_rf_text_in  | {'domain': 'in-domain (LIAR)',       'mode': 'binaire'})
rows.append(res_rf_text_out | {'domain': 'out-of-domain (WELFake)', 'mode': 'binaire'})

df_final = pd.DataFrame(rows)
os.makedirs('outputs', exist_ok=True)
df_final.to_csv('outputs/results_ood_binary.csv', index=False)
print("Résultats sauvegardés → outputs/results_ood_binary.csv")
df_final

Résultats sauvegardés → outputs/results_ood_binary.csv


,model,accuracy,f1_macro,f1_fake,f1_real,domain,mode
0,LR,0.7899,0.7867,0.7608,0.8126,in-domain (LIAR),binaire
1,LR,0.5000,0.3333,0.0000,0.6667,out-of-domain (WELFake),binaire
2,RF,0.7873,0.7837,0.7558,0.8117,in-domain (LIAR),binaire
3,RF,0.5000,0.3333,0.0000,0.6667,out-of-domain (WELFake),binaire
4,XGBoost,0.7747,0.7723,0.7493,0.7954,in-domain (LIAR),binaire
5,XGBoost,0.5004,0.3344,0.0020,0.6668,out-of-domain (WELFake),binaire
6,RF texte seul,0.6291,0.6151,0.5415,0.6886,in-domain (LIAR),binaire
7,RF texte seul,0.4683,0.4644,0.4185,0.5103,out-of-domain (WELFake),binaire
